# Hafta 9 — Yapay Sinir Ağlarına Giriş

Bu hafta ağı **kendimiz** kuruyoruz: nöron, perceptron kuralı, ileri yayılım, XOR. Geri yayılım (öğrenme) gelecek hafta; bugün yalnızca sklearn içeride yapacak.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
relu = lambda z: np.maximum(z, 0)
sigmoid = lambda z: 1/(1 + np.exp(-z))
def softmax(z): e = np.exp(z - z.max()); return e / e.sum()

## 1. Yapay nöron: tek satır

In [ ]:
x = np.array([1.0, 0.5]); w = np.array([1.8, 1.1]); b = -0.5
z = w @ x + b
print("z =", z, " sigmoid:", round(sigmoid(z), 3), " relu:", relu(z), " tanh:", round(np.tanh(z), 3))

## 2. Perceptron öğrenme kuralı (Örnek 9.1)

In [ ]:
def perceptron(X, y, eta=1.0, tur=20, yazdir=True):
    w = np.zeros(X.shape[1]); b = 0.0
    for t in range(tur):
        hata = 0
        for xi, yi in zip(X, y):
            yp = int(w @ xi + b >= 0)
            w += eta * (yi - yp) * xi; b += eta * (yi - yp); hata += int(yp != yi)
        if yazdir: print(f"tur {t+1:2d}: w={w}, b={b}, hata={hata}")
        if hata == 0: return w, b, True
    return w, b, False

X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
print("AND:"); w, b, ok = perceptron(X, np.array([0,0,0,1]))

In [ ]:
print("OR:"); perceptron(X, np.array([0,1,1,1]))
print("\nXOR:"); w, b, ok = perceptron(X, np.array([0,1,1,0]), tur=15, yazdir=False); print("yakınsadı mı?", ok, " son w, b:", w, b)

**Gözlem:** AND ve OR birkaç turda yakınsar; XOR asla — hata hiç sıfıra inmez. Doğrusal ayrılabilirlik teoreminin canlı gösterimi.

## 3. Aktivasyon fonksiyonları ve türevleri

In [ ]:
z = np.linspace(-5, 5, 200)
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, (f, df, ad) in zip(ax, ((sigmoid, lambda z: sigmoid(z)*(1-sigmoid(z)), "sigmoid"), (np.tanh, lambda z: 1-np.tanh(z)**2, "tanh"), (relu, lambda z: (z > 0)*1.0, "ReLU"))):
    a.plot(z, f(z), label="f"); a.plot(z, df(z), "--", label="f'"); a.set_title(ad); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()
print("sigmoid türevinin maksimumu:", (sigmoid(z)*(1-sigmoid(z))).max().round(3), "-> 20 katmanda 0.25^20 =", 0.25**20)

## 4. XOR'u elle çözmek: gizli katman

In [ ]:
def xor_agi(x):
    h = relu(np.array([[1, 1], [1, 1]]) @ x + np.array([-0.5, -1.0]))   # iki gizli ReLU nöron
    return int(np.array([1, -2]) @ h > 0.25), h
for xi in X:
    yp, h = xor_agi(xi); print(xi, "-> gizli", h, "-> çıktı", yp)

In [ ]:
# Gizli uzayda (h1, h2) noktalar artık doğrusal ayrılabilir
H = np.array([xor_agi(xi)[1] for xi in X]); y_xor = np.array([0,1,1,0])
plt.figure(figsize=(4, 3.5)); plt.scatter(H[y_xor==0, 0], H[y_xor==0, 1], s=120, label="0"); plt.scatter(H[y_xor==1, 0], H[y_xor==1, 1], s=120, marker="s", label="1")
hh = np.linspace(-.2, 1.7, 10); plt.plot(hh, (hh - 0.25)/2, "k--", label="h1 − 2h2 = 0.25"); plt.xlabel("h1"); plt.ylabel("h2"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. İleri yayılım: 2-3-1 ağ (Örnek 9.3)

In [ ]:
W1 = np.array([[1, -1], [0, 2], [1, 1.]]); b1 = np.array([0, -1, -1.]); W2 = np.array([[1, -2, 1.]]); b2 = np.array([0.5])
def ileri(x):
    z1 = W1 @ x + b1; h = relu(z1); z2 = W2 @ h + b2; return sigmoid(z2)[0], (z1, h, z2)
yp, (z1, h, z2) = ileri(np.array([1, 0.5]))
print("z1 =", z1, " h =", h, " z2 =", z2, " ŷ =", round(yp, 3), " log-loss(y=1) =", round(-np.log(yp), 3))
print("parametre sayısı:", W1.size + b1.size + W2.size + b2.size)

**Deneyin:** x = (0, 0) ve x = (2, −1) için ileri yayılımı çalıştırın. Hangi gizli nöronlar 'kapalı' (h = 0)?

## 6. Toplu (batch) ileri yayılım: tüm örnekler tek matris çarpımıyla

In [ ]:
Xb = np.array([[1, 0.5], [0, 0], [2, -1], [-1, 1]])           # 4 örnek × 2 özellik
H = relu(Xb @ W1.T + b1)                                      # 4 × 3
Yp = sigmoid(H @ W2.T + b2)                                   # 4 × 1
print(H); print(Yp.ravel().round(3))

## 7. Softmax (Örnek 9.2)

In [ ]:
print(softmax(np.array([2.0, 1.0, -1.0])).round(3), " toplam:", softmax(np.array([2.0, 1.0, -1.0])).sum())
print("sabit eklemek değiştirmez:", softmax(np.array([102.0, 101.0, 99.0])).round(3))

## 8. sklearn MLP: motor verisinde mimari karşılaştırması

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
m = pd.read_csv("motor_ariza.csv"); oz = ["calisma_saati", "titresim_rms_mms", "titresim_kurtosis", "sicaklik_C", "akim_dengesizlik_pct"]
Xtr, Xte, ytr, yte = train_test_split(m[oz], m.ariza, test_size=0.3, random_state=0, stratify=m.ariza)
for gizli in ((), (4,), (16,), (64,), (32, 32), (64, 64, 64)):
    mdl = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=gizli, max_iter=3000, random_state=0) if gizli else LogisticRegression())
    mdl.fit(Xtr, ytr)
    print(f"{str(gizli):14s} eğitim AUC={roc_auc_score(ytr, mdl.predict_proba(Xtr)[:,1]):.3f}  test AUC={roc_auc_score(yte, mdl.predict_proba(Xte)[:,1]):.3f}")

**Soru:** Gizli katman büyüdükçe eğitim AUC nasıl, test AUC nasıl değişiyor? Bu tablo 4. haftadaki hangi kavramı gösteriyor?

## 9. Evrensel yaklaşım deneyi

In [ ]:
from sklearn.neural_network import MLPRegressor
x = np.linspace(-2, 2, 300); hedef = np.sin(2*x) + 0.3*x
plt.figure(figsize=(10, 3)); plt.plot(x, hedef, "k", lw=1.5, label="hedef")
for n in (2, 5, 30):
    r = MLPRegressor(hidden_layer_sizes=(n,), activation="relu", max_iter=5000, random_state=1, learning_rate_init=0.01, tol=1e-6).fit(x[:, None], hedef)
    plt.plot(x, r.predict(x[:, None]), label=f"{n} nöron")
plt.legend(); plt.grid(alpha=.3); plt.show()

## 10. Alıştırmalar

**Alıştırma 1.** OR ve NAND kapılarını perceptron ile öğrenin; bulunan doğruları (w, b) 2-B grafikte çizin. NAND + OR + AND ile XOR'u üç perceptrondan kurun (XOR = AND(OR, NAND)).

In [ ]:
# Alıştırma 1

**Alıştırma 2.** `ileri` fonksiyonunu genel hâle getirin: katman boyutları listesi (ör. [2, 8, 4, 1]) verilince rastgele ağırlıklarla ağı kursun ve ileri yayılım yapsın. [10, 64, 32, 4] için parametre sayısını fonksiyonla doğrulayın (Örnek 9.4: 2916).

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Aktivasyonsuz 3 katmanlı ağın tek doğrusal katmana eşdeğer olduğunu sayısal olarak gösterin: rastgele W₁, W₂, W₃ ile ağı kurun, W = W₃W₂W₁ hesaplayın, 5 rastgele girdi için iki hesabın aynı sonucu verdiğini doğrulayın.

In [ ]:
# Alıştırma 3

**Alıştırma 4.** Sigmoid gizli katmanlı MLPClassifier (activation='logistic') ile ReLU'lu olanı 3, 10 ve 20 gizli katman (her biri 32 nöron, alpha=0.01) için karşılaştırın: `n_iter_`, son kayıp (`loss_`) ve test AUC. Kaybolan gradyan hangi derinlikte, hangi aktivasyonda ortaya çıkıyor?

In [ ]:
# Alıştırma 4

**Alıştırma 5.** Evrensel yaklaşım deneyinde 30 nöronlu ağı 4. haftadaki gibi değerlendirin: veriyi eğitim/test böl, hedefe σ = 0.2 gürültü ekle, nöron sayısı 2-200 için eğitim/test RMSE eğrisini çizin. Aşırı öğrenme nerede başlıyor?

In [ ]:
# Alıştırma 5